#### LIBRARY IMPORTS

In [11]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Neural Netowrk specifc imports
import copy # for deepcopy in save_results()

import torch
import torch.nn as nn # NN layers and loss functions
import torch.optim as optim # Optimization Algorithms
# Batching Data:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

#### CONFIGURATOINS

In [12]:
FEATURE_ROOT = "speaker_wise-eGeMAPs/functionals/MP4_raw/"
OUTPUT_ROOT = "model_outputs/MP4_raw/neural-network"
RANDOM_SEED = 37
THRESHOLD = 0.5
# TRAIN_RATIO = 0.8
# VAL_RATIO = 0.1
# TEST_RATIO = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHILD_SPEAKER = {
     
    "speaker_functional_p5-s2.csv": "SPEAKER_00",
    "speaker_functional_p5-s7.csv": "SPEAKER_01",
    "speaker_functional_p5-s8.csv": "SPEAKER_05",
    "speaker_functional_p5-s10.csv": "SPEAKER_01",
    "speaker_functional_p5-s13.csv": "SPEAKER_01",

    "speaker_functional_p7-s5.csv": "SPEAKER_01",
    "speaker_functional_p7-s6.csv": "SPEAKER_08",
    "speaker_functional_p7-s7.csv": "SPEAKER_02",
    "speaker_functional_p7-s8.csv": "SPEAKER_00",
    "speaker_functional_p7-s16.csv": "SPEAKER_00",
    "speaker_functional_p7-s17.csv": "SPEAKER_00",
    "speaker_functional_p7-s18.csv": "SPEAKER_02",
    "speaker_functional_p7-s29.csv": "SPEAKER_00",

    "speaker_functional_p9-s3-1.csv": "SPEAKER_01",
    "speaker_functional_p9-s3-2.csv": "SPEAKER_04",
    "speaker_functional_p9-s4.csv": "SPEAKER_06",
    "speaker_functional_p9-s9.csv": "SPEAKER_03",
    "speaker_functional_p9-s15.csv": "SPEAKER_01",

    "speaker_functional_p11-s2.csv": "SPEAKER_02",
    "speaker_functional_p11-s4.csv": "SPEAKER_01",
    "speaker_functional_p11-s8.csv": "SPEAKER_03",
    "speaker_functional_p11-s9.csv": "SPEAKER_00",
    "speaker_functional_p11-s11.csv": "SPEAKER_05",
    "speaker_functional_p11-s15.csv": "SPEAKER_00",
    "speaker_functional_p11-s16-2.csv": "SPEAKER_01",
    "speaker_functional_p11-s19.csv": "SPEAKER_00",
    "speaker_functional_p11-s22-2.csv": "SPEAKER_02",

    "speaker_functional_p12-s2-2.csv": "SPEAKER_00",
    "speaker_functional_p12-s3.csv": "SPEAKER_01",
    "speaker_functional_p12-s6.csv": "SPEAKER_01",
    "speaker_functional_p12-s8.csv": "SPEAKER_00",
    "speaker_functional_p12-s10.csv": "SPEAKER_03",

    "speaker_functional_p17-s2.csv": "SPEAKER_01",
    "speaker_functional_p17-s3.csv": "SPEAKER_04",
    "speaker_functional_p17-s5.csv": "SPEAKER_04",
    "speaker_functional_p17-s6.csv": "SPEAKER_02",

    "speaker_functional_p18-s3.csv": "SPEAKER_00",
    "speaker_functional_p18-s4.csv": "SPEAKER_01",
    "speaker_functional_p18-s5.csv": "SPEAKER_00",
    "speaker_functional_p18-s7.csv": "SPEAKER_01",
    "speaker_functional_p18-s8.csv": "SPEAKER_00",
    "speaker_functional_p18-s9.csv": "SPEAKER_01",
    "speaker_functional_p18-s10.csv": "SPEAKER_06",
    "speaker_functional_p18-s11.csv": "SPEAKER_00",
    "speaker_functional_p18-s12.csv": "SPEAKER_00",
    "speaker_functional_p18-s13.csv": "SPEAKER_01",
    "speaker_functional_p18-s15.csv": "SPEAKER_02",
    "speaker_functional_p18-s17.csv": "SPEAKER_01",
    "speaker_functional_p18-s18.csv": "SPEAKER_00",
    "speaker_functional_p18-s19.csv": "SPEAKER_02",
    "speaker_functional_p18-s20.csv": "SPEAKER_01",

}

#### FUNCTIONS

In [13]:
# Early stopping
PATIENCE = 10 # Stop after 10 epochs of no improvement in validation loss

In [14]:
def load_participant_data(participant_folder):
    # Load all CSV files for a given participant folder in a sorted fashion
    csv_files = sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv")) # For multiple files

    #  Function to filter the child speaker from single csv file
    def load_single_speaker(csv_path):
        df = pd.read_csv(csv_path)    
        filename = os.path.basename(csv_path)

        # Error handle
        if filename not in CHILD_SPEAKER:
            raise ValueError(
                f"No child speaker mapping for {filename}"
            )
        
        target_speaker = CHILD_SPEAKER[filename]
        
        # df = df[True] where it is True for child speaker
        df = df[df["speaker"] == target_speaker]

        return df

    ### DEBUG STATEMENT   
    print(f"{participant_folder}: {len(csv_files)} CSV files")
    ###

    # Load every session for this participant
    full_df = pd.concat(
        [
            # Load child speaker only from file f
            load_single_speaker(f)
            for f in csv_files
        ],
        ignore_index=True
    )
     
    ### DEBUG STATEMENT
    print(f"Total child utterances: {len(full_df)}")
    ### 

    # Remove unnecessary columns from eGeMAPs table
    drop_cols = [
        "participant",
        "session",
        "clip_id",
        "speaker",

        "engagement_start_time",
        "engagement_end_time",

        "speaker_start_time",
        "speaker_end_time",

        "num_segments",
        "speech_duration",
    ]

    full_df = full_df.drop(
        columns=[c for c in drop_cols if c in full_df.columns]
    )

    print(full_df.columns)

    # Separate Features and Labels
    X = full_df.drop(columns=["label"])
    y = full_df["label"]

    # We reset the index below because we'ev filtered non-child speaker rows
    return (
        X.reset_index(drop=True),
        y.reset_index(drop=True)
    )

In [15]:
def preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_val = encoder.transform(y_val)
    y_test = encoder.transform(y_test)

    ### DEBUG STATEMENT
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))
    ###
    
    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_val = scaler.transform(X_val) # Transform the data using the learned mean and std
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        scaler,
        encoder
    )  

In [16]:
class NeuralNetwork(nn.Module):

    # Define the architecture of the neural network ; Constructor
    def __init__(self, input_dim):

        super().__init__() # Initialize the parent class (nn.Module) first, then inherit functionalities

        self.network = nn.Sequential(
            # First Hidden Layer 88 -> 128 nueruons
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Second Hidden Layer 128 -> 64 nueruons
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output Layer 64 -> 1 nuerons
            nn.Linear(64, 1)

        )
    
    # Forward pass through the network; Automatically called  when calling model; Then return the model predictions
    def forward(self, x):
        return self.network(x)


# Build the model, then move it to GPU/ CPU and print the model architecture
def build_model(input_dim):
    # Build model
    model = NeuralNetwork(input_dim)
    
    # Move model to GPU/ CPU
    model.to(DEVICE)
    
    print(model)

    return model

In [17]:
def train_model(
        model,
        train_loader,
        val_loader
):
    # BCEWithLogitsLoss is good for our case of binary classification
    # It performs sigmoid + Binary Cross Entropy Loss 
    criterion = nn.BCEWithLogitsLoss()

    # Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Initialize best loss to be infinity and patience counter to 0
    best_loss = float("inf")
    patience_counter = 0

    # Store training and validation loss for each epoch (perhaps for plotting later)
    history = {
        "train_loss": [],
        "val_loss": []
    }

    # Epoch Loop; Max is 100, but may stop earlier due to early stopping
    for epoch in range(100):
        
        # TRAINING STEPS:
        # Activate training mode (Dropout layers and gradient computation)
        model.train()
        train_loss = 0

        # Train Batch-wise (32)
        for X_batch, y_batch in train_loader:
            
            # Move batch to GPU/ CPU
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            # Reset the gradients before backpropagation
            optimizer.zero_grad()

            # Foeward pass prediction
            outputs = model(X_batch)

            # Compute loss
            loss = criterion(outputs, y_batch)

            # Backpropagation 
            # Computes gradients
            loss.backward()
            # Update weights using optimizer
            optimizer.step()
            # Accumute training loss
            train_loss += loss.item() # loss is a tensor

        # Average traingin loss per batch
        train_loss /= len(train_loader)

        # VALIDATION STEPS
        # Turn off training mode( No Dropout layers and gradient computation)
        model.eval()
        val_loss = 0

        # Disable gradient computation
        with torch.no_grad():
            # Validate Batch-wise (32)
            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
 
                # Calls the forward pass of the model to get predictions
                outputs = model(X_batch)
                # Compute validation loss; criterion is the loss function defined in above 
                loss = criterion(outputs, y_batch)
                #Accumulate validation loss
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"Train={train_loss:.4f} "
            f"Val={val_loss:.4f}"
        )

        # If validation loss improves, save the model weights and reset patience counter
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0

        # If validation loss does not improve, increment patience counter and check for early stopping
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping")
                # Exit training loop
                break

    # Restore the best model weights after training is complete
    model.load_state_dict(best_weights)

    return history

In [18]:
def evaluate_model(
        model,
        test_loader,
        encoder
):
    # Turn off training mode( No Dropout layers and gradient computation)
    model.eval()

    probabilities = [] # Store the predicted probabilities for the positive class (engaged)
    predictions = [] # Store the predicted class labels (0 or 1)
    actual = [] # Store the actual class labels (0 or 1)

    # Disable gradient computation
    with torch.no_grad():
        # Exaluate Batch-wise (32)
        for X_batch, y_batch in test_loader:
            
            X_batch = X_batch.to(DEVICE)
            # Forward pass prediction
            outputs = model(X_batch)
            # Apply sigmoid to get probabilities (0 to 1)
            probs = torch.sigmoid(outputs)
            # Convert probabilities to binary predictions (0 or 1) using a threshold of 0.5
            preds = (probs >= THRESHOLD).float()

            # NumPy cannotread GPU tensors, so we need to move them to CPU and convert to NumPy arrays before storing them in lists
            probabilities.extend(probs.cpu().numpy().flatten())
            predictions.extend(preds.cpu().numpy().flatten())
            actual.extend(y_batch.numpy().flatten())

    # Convert lists to NumPy arrays and ensure they are of integer type
    probabilities = np.array(probabilities).astype(float)
    predictions = np.array(predictions).astype(int)
    actual = np.array(actual).astype(int)

    # DEBUGGING STATEMENTS
    # How many samples are actually there from each class
    print("\nActual class counts:")
    print(pd.Series(actual).value_counts()) 
    # How many samples were predicted from each class
    print("\nPredicted class counts:")
    print(pd.Series(predictions).value_counts())
    # Print first 20 probabilty values
    print("\nPredicted probabilities:")
    print(probabilities[:20])

    print(f"\nMinimum probability : {probabilities.min():.3f}")
    print(f"Maximum probability : {probabilities.max():.3f}")
    print(f"Average probability : {probabilities.mean():.3f}")

    # actual contains the true labels; filter the engaged and disengaged probabilities based on the actual labels
    engaged_probs = probabilities[actual == 1]
    disengaged_probs = probabilities[actual == 0]

    print("\nProbability Statistics")
    print("----------------------")
    print(f"Engaged mean      : {engaged_probs.mean():.3f}")
    print(f"Engaged std       : {engaged_probs.std():.3f}")
    print(f"Disengaged mean   : {disengaged_probs.mean():.3f}")
    print(f"Disengaged std    : {disengaged_probs.std():.3f}")

    # plt.figure(figsize=(6,4))
    # plt.hist(engaged_probs, bins=15, alpha=0.6, label="Engaged")
    # plt.hist(disengaged_probs, bins=15, alpha=0.6, label="Disengaged")
    # plt.xlabel("Predicted Probability")
    # plt.ylabel("Count")
    # plt.legend()
    # plt.show()

    # Confusion Matrix
    cm = confusion_matrix(actual, predictions)

    # Compute metrics
    metrics_dictionary = {
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1 Score": f1_score(actual, predictions)
    }

    report = classification_report(
        actual,
        predictions,
        target_names=encoder.classes_,
        output_dict=True # Return the report as a dictionary instead of a string to make csv
    )

    print(classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    ))

    return {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": actual,
        "confusion_matrix": cm,
        "classification_report": report
    }

In [19]:
''' The following are saved:
    Model weights (model.pt)
    Scaler (scaler.pkl)
    Encoder (encoder.pkl)
    Training History (history.csv)
    Metrics (metrics.csv)
    Confusion Matrix (confusion_matrix.csv)
    Classification Report (classification_report.csv) '''
    
def save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    ):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    torch.save(model.state_dict(),participant_output / "neural-network.pt")
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save History
    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(participant_output, "history.csv"),index=False)
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [20]:
summary_results = [] # Store average metircs for each participant
participants = sorted(os.listdir(FEATURE_ROOT))
all_fold_results = [] # Store metrics for each fold of each participant

# Loop through each participant and train a model for each participant
for participant in participants:

    print(f"Training {participant}")
    # Load ALL data for this participant; X= GeMAPs features, y= labels (engaged/disengaged)
    X, y = load_participant_data(participant)

    skf = StratifiedKFold(
        n_splits=5, # 5 folds
        shuffle=True, # randomise the samples before splirtting
        random_state=RANDOM_SEED
    )
    # Store metrics for each fold of this participant
    participant_metrics = []

    # Loop through each fold; train_idx= indices for training samples, test_idx= indices for testing samples
    # 80% train, 20% test;
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        print(f"\nFold {fold}/5")
        # Create Train/Test feature set based on train indexes for current fold
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        # Create Train/Test label set based on train indexes for current fold
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Split again: 80% of train (=64%) used for training, 20% of train (=16%) used for validation
        X_train, X_val, y_train, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.20,
            stratify=y_train, # Maintain class distribution
            random_state=RANDOM_SEED
        )

        # DEBUGGING STATEMENTS
        print("\nTraining class distribution:")
        print(y_train.value_counts())

        print("\nValidation class distribution:")
        print(y_val.value_counts())

        print("\nTesting class distribution:")
        print(y_test.value_counts())
        ###

        # Preprocess data
        X_train, X_val, X_test, y_train, y_val, y_test, scaler, encoder = preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test)

        # Convert NumPy arrays into PyTorch tensors.
        X_train_tensor = torch.FloatTensor(X_train)
        X_val_tensor = torch.FloatTensor(X_val)
        X_test_tensor = torch.FloatTensor(X_test)

        # Shape of y_train, y_val, y_test is (N,), but we need (N,1) for BCEWithLogitsLoss
        y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
        y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1)
        y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

        # Create TensorDatasets; Join features with corresponding labels
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

        # Create DataLoaders; Batch the data (32) and randomly shuffle
        # FOr trainging dataset
        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True # Shuffle training data for better generalization
        )
        # FOr validation dataset
        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False
        )
        # FOr testing dataset
        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            shuffle=False
        )

        # Build model
        model = build_model(X_train.shape[1])

        # Train model and return training history
        history = train_model(model, train_loader, val_loader)

        # Evaluate model and return evaluation metrics
        evaluation = evaluate_model(model, test_loader, encoder)

        # Append metrics for this participant to the summary results
        participant_metrics.append(evaluation["metrics"])

        # Append fold results to all_fold_results
        all_fold_results.append({
            "Participant": participant,
            "Fold": fold,
            # evaluation["metrics"] is a dictionary containing the metrics for this fold, ** unpacks the dictionary 
            **evaluation["metrics"]
        })
        
        # Save results
        save_results(
            f"{participant}/fold_{fold}",
            model,
            scaler,
            encoder,
            history,
            evaluation
        )
    
    metrics_df = pd.DataFrame(participant_metrics)
    
    summary_results.append({
        "Participant": participant,

        "Accuracy Mean": metrics_df["Accuracy"].mean(),
        "Accuracy Std": metrics_df["Accuracy"].std(),

        "Precision Mean": metrics_df["Precision"].mean(),
        "Precision Std": metrics_df["Precision"].std(),

        "Recall Mean": metrics_df["Recall"].mean(),
        "Recall Std": metrics_df["Recall"].std(),

        "F1 Mean": metrics_df["F1 Score"].mean(),
        "F1 Std": metrics_df["F1 Score"].std()
    })
      
    pd.DataFrame(all_fold_results).to_csv(
        "fold_results.csv",
        index=False
    )

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)
    

Training p11
p11: 9 CSV files
Total child utterances: 398
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean',
       'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0',
       'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0',
       'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope',
       'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope',
       'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean',
       'spectralFlux_sma3_stddevNorm', '

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=88, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)
Epoch 1 Train=0.6859 Val=0.7052
Epoch 2 Train=0.6654 Val=0.7384
Epoch 3 Train=0.6652 Val=0.7574
Epoch 4 Train=0.6291 Val=0.7632
Epoch 5 Train=0.6180 Val=0.7954
Epoch 6 Train=0.6045 Val=0.8132
Epoch 7 Train=0.5948 Val=0.8397
Epoch 8 Train=0.5818 Val=0.8525
Epoch 9 Train=0.5539 Val=0.8701
Epoch 10 Train=0.5269 Val=0.8978
Epoch 11 Train=0.5098 Val=0.9614
Early stopping

Actual class counts:
1    54
0    41
Name: count, dtype: int64

Predicted class counts:
1    83
0    12
Name: count, dtype: int64

Predicted probabilities:
[0.57380891 0.60543621 0.54395157 0.5609017  0.58166766 0.58004087
 0.58114189 0.60462856 0.55365431 0.61443675 0.59123766 0.61433059
 0.

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 3 Train=0.6517 Val=0.6186
Epoch 4 Train=0.6371 Val=0.6025
Epoch 5 Train=0.6194 Val=0.5874
Epoch 6 Train=0.5871 Val=0.5744
Epoch 7 Train=0.5806 Val=0.5631
Epoch 8 Train=0.5632 Val=0.5544
Epoch 9 Train=0.5490 Val=0.5478
Epoch 10 Train=0.5221 Val=0.5439
Epoch 11 Train=0.4954 Val=0.5418
Epoch 12 Train=0.4930 Val=0.5412
Epoch 13 Train=0.4687 Val=0.5402
Epoch 14 Train=0.4646 Val=0.5403
Epoch 15 Train=0.4279 Val=0.5407
Epoch 16 Train=0.4144 Val=0.5419
Epoch 17 Train=0.4008 Val=0.5433
Epoch 18 Train=0.3857 Val=0.5457
Epoch 19 Train=0.3699 Val=0.5513
Epoch 20 Train=0.3440 Val=0.5589
Epoch 21 Train=0.3243 Val=0.5681
Epoch 22 Train=0.3090 Val=0.5810
Epoch 23 Train=0.3025 Val=0.5976
Early stopping

Actual class counts:
1    25
0    13
Name: count, dtype: int64

Predicted class counts:
1    33
0     5
Name: count, dtype: int64

Predicted probabilities:
[0.75565976 0.86621553 0.74007618 0.90435314 0.46010178 0.42046532
 0.52884394 0.6230697  0.95428663 0.92906421 0.83732265 0.87823296
 0.72112

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 1 Train=0.6821 Val=0.6486
Epoch 2 Train=0.6553 Val=0.6281
Epoch 3 Train=0.6423 Val=0.6080
Epoch 4 Train=0.6300 Val=0.5900
Epoch 5 Train=0.5944 Val=0.5756
Epoch 6 Train=0.5770 Val=0.5630
Epoch 7 Train=0.5792 Val=0.5491
Epoch 8 Train=0.5710 Val=0.5388
Epoch 9 Train=0.5298 Val=0.5289
Epoch 10 Train=0.5201 Val=0.5217
Epoch 11 Train=0.5148 Val=0.5150
Epoch 12 Train=0.5050 Val=0.5075
Epoch 13 Train=0.5003 Val=0.4989
Epoch 14 Train=0.4649 Val=0.4879
Epoch 15 Train=0.4688 Val=0.4772
Epoch 16 Train=0.4095 Val=0.4667
Epoch 17 Train=0.4005 Val=0.4573
Epoch 18 Train=0.3828 Val=0.4492
Epoch 19 Train=0.3935 Val=0.4415
Epoch 20 Train=0.3570 Val=0.4332
Epoch 21 Train=0.2863 Val=0.4238
Epoch 22 Train=0.2792 Val=0.4141
Epoch 23 Train=0.3134 Val=0.4085
Epoch 24 Train=0.2562 Val=0.4062
Epoch 25 Train=0.2400 Val=0.3973
Epoch 26 Train=0.2368 Val=0.3920
Epoch 27 Train=0.1879 Val=0.3807
Epoch 28 Train=0.1919 Val=0.3697
Epoch 29 Train=0.1759 Val=0.3643
Epoch 30 Train=0.1651 Val=0.3606
Epoch 31 Train=0.17

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 10 Train=0.4834 Val=0.7170
Epoch 11 Train=0.5134 Val=0.7463
Epoch 12 Train=0.4590 Val=0.7795
Epoch 13 Train=0.4848 Val=0.8194
Epoch 14 Train=0.4844 Val=0.8558
Epoch 15 Train=0.3929 Val=0.8922
Early stopping

Actual class counts:
1    13
0     9
Name: count, dtype: int64

Predicted class counts:
1    19
0     3
Name: count, dtype: int64

Predicted probabilities:
[0.5968039  0.58678043 0.84388268 0.6174683  0.64593709 0.75898737
 0.63124591 0.62789065 0.5263499  0.68356973 0.52328545 0.53072125
 0.46403092 0.52325159 0.62286144 0.52029765 0.74673408 0.51262969
 0.51146996 0.48104054]

Minimum probability : 0.464
Maximum probability : 0.844
Average probability : 0.595

Probability Statistics
----------------------
Engaged mean      : 0.609
Engaged std       : 0.082
Disengaged mean   : 0.576
Disengaged std    : 0.114
              precision    recall  f1-score   support

  disengaged       1.00      0.33      0.50         9
     engaged       0.68      1.00      0.81        13

    a